# Eigenvalues and Eigenvectors Uygulaması

Bu notebookta, verilen referans çalışmaya benzer şekilde hazır `numpy.linalg.eig` fonksiyonu kullanılmadan özdeğer hesabı uygulanmıştır. Daha sonra aynı matris üzerinde `numpy.linalg.eig` kullanılarak sonuçlar karşılaştırılmıştır.

In [2]:
import numpy as np

In [3]:
# Referans repodaki örnek matrisi kullandım
A = np.array([
    [6,  1, -1],
    [0,  7,  0],
    [3, -1,  2]
], dtype=float)

print("Matris A:")
print(A)

Matris A:
[[ 6.  1. -1.]
 [ 0.  7.  0.]
 [ 3. -1.  2.]]


In [4]:
def poly_topla(p1, p2, isaret=1):
    """
    Polinomları toplar.
    Katsayılar küçük dereceden büyük dereceye doğru tutulur.
    Örnek: [2, 3, 1] = 2 + 3x + x^2
    """
    uzunluk = max(len(p1), len(p2))
    p1 = p1 + [0] * (uzunluk - len(p1))
    p2 = p2 + [0] * (uzunluk - len(p2))
    return [a + isaret * b for a, b in zip(p1, p2)]


def poly_carp(p1, p2):
    """
    İki polinomu çarpar.
    """
    sonuc = [0] * (len(p1) + len(p2) - 1)
    for i in range(len(p1)):
        for j in range(len(p2)):
            sonuc[i + j] += p1[i] * p2[j]
    return sonuc


def minor_al(matris, satir, sutun):
    """
    Verilen satır ve sütun silinerek minor matris oluşturulur.
    """
    yeni = []
    for i in range(len(matris)):
        if i == satir:
            continue
        yeni_satir = []
        for j in range(len(matris[i])):
            if j == sutun:
                continue
            yeni_satir.append(matris[i][j])
        yeni.append(yeni_satir)
    return yeni


def determinant_polinomu(poly_matrix):
    """
    Elemanları polinom olan bir matrisin determinant polinomunu döndürür.
    """
    n = len(poly_matrix)

    if n == 1:
        return poly_matrix[0][0]

    if n == 2:
        sol = poly_carp(poly_matrix[0][0], poly_matrix[1][1])
        sag = poly_carp(poly_matrix[0][1], poly_matrix[1][0])
        return poly_topla(sol, sag, isaret=-1)

    sonuc = [0]

    for j in range(n):
        eleman = poly_matrix[0][j]
        alt_matris = minor_al(poly_matrix, 0, j)
        alt_det = determinant_polinomu(alt_matris)
        terim = poly_carp(eleman, alt_det)

        if j % 2 == 0:
            sonuc = poly_topla(sonuc, terim, isaret=1)
        else:
            sonuc = poly_topla(sonuc, terim, isaret=-1)

    return sonuc


def karakteristik_polinom_matrisi(matris):
    """
    A - λI matrisini polinom biçiminde kurar.
    Diyagonal elemanlar: [a_ii, -1]
    Diyagonal dışı elemanlar: [a_ij]
    """
    n = len(matris)
    poly_matrix = []

    for i in range(n):
        satir = []
        for j in range(n):
            if i == j:
                satir.append([float(matris[i][j]), -1.0])
            else:
                satir.append([float(matris[i][j])])
        poly_matrix.append(satir)

    return poly_matrix


def manuel_ozdegerler(matris):
    """
    Karakteristik polinomu bulup köklerini hesaplar.
    """
    poly_matrix = karakteristik_polinom_matrisi(matris)
    katsayilar_kucukten_buyuge = determinant_polinomu(poly_matrix)

    # np.roots büyük dereceden küçük dereceye doğru katsayı ister
    katsayilar_buyukten_kucuge = katsayilar_kucukten_buyuge[::-1]

    kokler = np.roots(katsayilar_buyukten_kucuge)
    return katsayilar_kucukten_buyuge, kokler

In [5]:
char_poly, manual_vals = manuel_ozdegerler(A)

print("Karakteristik polinom katsayıları (küçük dereceden büyük dereceye):")
print(char_poly)

print("\nManuel yöntemle bulunan özdeğerler:")
print(manual_vals)

Karakteristik polinom katsayıları (küçük dereceden büyük dereceye):
[105.0, -71.0, 15.0, -1.0]

Manuel yöntemle bulunan özdeğerler:
[7. 5. 3.]


In [6]:
np_vals, np_vecs = np.linalg.eig(A)

print("NumPy ile bulunan özdeğerler:")
print(np_vals)

print("\nNumPy ile bulunan özvektörler:")
print(np_vecs)

NumPy ile bulunan özdeğerler:
[5. 3. 7.]

NumPy ile bulunan özvektörler:
[[0.70710678 0.31622777 0.58834841]
 [0.         0.         0.78446454]
 [0.70710678 0.9486833  0.19611614]]


In [7]:
manual_sorted = np.sort_complex(manual_vals)
numpy_sorted = np.sort_complex(np_vals)

print("Sıralanmış manuel özdeğerler:")
print(manual_sorted)

print("\nSıralanmış NumPy özdeğerleri:")
print(numpy_sorted)

print("\nFark:")
print(manual_sorted - numpy_sorted)

print("\nYakın mı kontrolü:")
print(np.allclose(manual_sorted, numpy_sorted))

Sıralanmış manuel özdeğerler:
[3.+0.j 5.+0.j 7.+0.j]

Sıralanmış NumPy özdeğerleri:
[3.+0.j 5.+0.j 7.+0.j]

Fark:
[ 2.22044605e-15+0.j -1.68753900e-14+0.j  1.06581410e-14+0.j]

Yakın mı kontrolü:
True


In [8]:
print("NumPy'den gelen özvektörlerin kontrolü:\n")

for i in range(len(np_vals)):
    sol = A @ np_vecs[:, i]
    sag = np_vals[i] * np_vecs[:, i]

    print(f"{i+1}. özdeğer için kontrol:")
    print("A @ v = ", sol)
    print("lambda * v = ", sag)
    print("Yakın mı?: ", np.allclose(sol, sag))
    print("-" * 40)

NumPy'den gelen özvektörlerin kontrolü:

1. özdeğer için kontrol:
A @ v =  [3.53553391 0.         3.53553391]
lambda * v =  [3.53553391 0.         3.53553391]
Yakın mı?:  True
----------------------------------------
2. özdeğer için kontrol:
A @ v =  [0.9486833  0.         2.84604989]
lambda * v =  [0.9486833  0.         2.84604989]
Yakın mı?:  True
----------------------------------------
3. özdeğer için kontrol:
A @ v =  [4.11843884 5.49125178 1.37281295]
lambda * v =  [4.11843884 5.49125178 1.37281295]
Yakın mı?:  True
----------------------------------------


## Kısa Yorum

Bu uygulamada, referans çalışmaya benzer şekilde karakteristik polinom üzerinden manuel özdeğer hesabı yapılmıştır. Daha sonra aynı matris için `numpy.linalg.eig` kullanılmıştır. Sonuçlar karşılaştırıldığında özdeğerlerin aynı ya da çok yakın olduğu görülmüştür. Küçük farklar varsa bunlar sayısal gösterimden kaynaklanmaktadır.

Manuel yöntem konunun mantığını anlamak açısından faydalıdır. Ancak pratikte daha kısa, daha güvenilir ve daha kullanışlı olduğu için `numpy.linalg.eig` fonksiyonunu kullanmak daha uygundur.